# PAD Model Implementation

Transformer-based model for anomaly detection and precursor-of-anomaly (PoA) prediction on irregular time series.

## Configuration

Edit these parameters before running:
- WINDOW_SIZE: Events per window (128)
- STEP_SIZE: Sliding window step (64)
- NUM_LAYERS: Transformer layers (2)
- NUM_EPOCHS: Training epochs (10)
- LEARNING_RATE: Learning rate (1e-4)
- BATCH_SIZE: Batch size (16)
- ANOMALY_PERCENTILE: Detection threshold (95)

In [11]:
# ======================
# CONFIG
# ======================

# Data parameters
WINDOW_SIZE = 128
STEP_SIZE = 64
DATA_PATH = 'data/process_metrics_0202_0903_CED.csv'

# Model architecture
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 256
DROPOUT = 0.1
USE_SHARED = True

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 300
TRAIN_RATIO = 0.8

# Evaluation
ANOMALY_PERCENTILE = 95

print('Configuration loaded')

Configuration loaded


## Section 1: Dataset Preparation

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TimeEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        self.delta_mlp = nn.Sequential(
            nn.Linear(1, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )
    
    def forward(self, t_abs, t_delta):
        return self.pe[t_abs] + self.delta_mlp(t_delta)

class InputEmbedding(nn.Module):
    def __init__(self, input_dim, d_model, max_len=5000):
        super().__init__()
        self.value_linear = nn.Linear(input_dim, d_model)
        self.time_encoding = TimeEncoding(d_model, max_len)
    
    def forward(self, x, t_abs, t_delta):
        return self.value_linear(x) + self.time_encoding(t_abs, t_delta)

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward=512, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
    
    def forward(self, tokens, mask=None):
        return self.encoder(tokens, src_key_padding_mask=mask)

class PADTransformer(nn.Module):
    def __init__(
        self, input_dim, d_model=128, nhead=8, num_layers=4,
        dim_feedforward=512, dropout=0.1, max_len=5000, use_shared=True
    ):
        super().__init__()
        self.use_shared = use_shared
        self.d_model = d_model
        
        self.embedding = InputEmbedding(input_dim + 1, d_model, max_len)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.anomaly_encoder = TransformerEncoder(d_model, nhead, num_layers, dim_feedforward, dropout)
        self.poa_encoder = TransformerEncoder(d_model, nhead, num_layers, dim_feedforward, dropout)
        
        if use_shared:
            self.shared_encoder = TransformerEncoder(d_model, nhead, num_layers, dim_feedforward, dropout)
        
        self.anomaly_head = nn.Linear(d_model, 1)
        self.poa_head = nn.Linear(d_model, 1)
    
    def embed_window(self, x, delta_t, t_abs):
        x_with_delta = torch.cat([x, delta_t], dim=-1)
        return self.embedding(x_with_delta, t_abs.squeeze(-1), delta_t)
    
    def transformer_forward(self, tokens, branch='anomaly'):
        batch_size = tokens.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens_with_cls = torch.cat([cls_tokens, tokens], dim=1)
        h = self.anomaly_encoder(tokens_with_cls) if branch == 'anomaly' else self.poa_encoder(tokens_with_cls)
        if self.use_shared:
            h = self.shared_encoder(h)
        return h[:, 0, :]
    
    def forward(self, w_i, w_i_plus_1):
        tokens_i = self.embed_window(w_i['x'], w_i['delta_t'], w_i['t_abs'])
        tokens_i_plus_1 = self.embed_window(w_i_plus_1['x'], w_i_plus_1['delta_t'], w_i_plus_1['t_abs'])
        h_i = self.transformer_forward(tokens_i, 'anomaly')
        h_i_plus_1 = self.transformer_forward(tokens_i_plus_1, 'anomaly')
        z_i = self.transformer_forward(tokens_i, 'poa')
        # Return logits (no sigmoid) for numerical stability
        return self.anomaly_head(h_i), self.anomaly_head(h_i_plus_1), self.poa_head(z_i)

print('Model classes defined')

Data preparation functions defined


In [13]:
from tqdm import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

class PADDataset(torch.utils.data.Dataset):
    def __init__(self, windows_norm):
        self.windows = windows_norm
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return {
            'x': torch.FloatTensor(w['x']),
            'delta_t': torch.FloatTensor(w['delta_t']),
            't_abs': torch.LongTensor(w['t_abs'])
        }

dataset = PADDataset(windows_norm)
train_size = int(TRAIN_RATIO * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True if device.type == 'cuda' else False)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True if device.type == 'cuda' else False)

model = PADTransformer(
    input_dim=wide_df.shape[1],
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    use_shared=USE_SHARED
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Train: {train_size}, Val: {val_size}')

history = {'train': [], 'val': [], 'lr': [], 'epoch_time': []}
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=False)
    
    for batch in train_pbar:
        w_i = {
            'x': batch['x'].to(device),
            'delta_t': batch['delta_t'].to(device),
            't_abs': batch['t_abs'].to(device)
        }
        w_i_plus_1 = w_i
        
        logits_a_i, logits_a_i_plus_1, logits_p_i_plus_1 = model(w_i, w_i_plus_1)
        
        # Use BCEWithLogitsLoss for numerical stability
        L_a = F.binary_cross_entropy_with_logits(logits_a_i.squeeze(), torch.zeros_like(logits_a_i.squeeze()))
        L_kd = F.binary_cross_entropy_with_logits(logits_a_i_plus_1.squeeze(), torch.sigmoid(logits_p_i_plus_1.squeeze()).detach())
        loss = L_a + L_kd
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]', leave=False)
    
    with torch.no_grad():
        for batch in val_pbar:
            w_i = {
                'x': batch['x'].to(device),
                'delta_t': batch['delta_t'].to(device),
                't_abs': batch['t_abs'].to(device)
            }
            w_i_plus_1 = w_i
            logits_a_i, logits_a_i_plus_1, logits_p_i_plus_1 = model(w_i, w_i_plus_1)
            L_a = F.binary_cross_entropy_with_logits(logits_a_i.squeeze(), torch.zeros_like(logits_a_i.squeeze()))
            L_kd = F.binary_cross_entropy_with_logits(logits_a_i_plus_1.squeeze(), torch.sigmoid(logits_p_i_plus_1.squeeze()).detach())
            batch_loss = (L_a + L_kd).item()
            val_loss += batch_loss
            val_pbar.set_postfix({'loss': f'{batch_loss:.4f}'})
    
    val_loss /= len(val_loader)
    epoch_time = time.time() - epoch_start
    
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['lr'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s | *BEST*')
    else:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s')

print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Total training time: {sum(history["epoch_time"]):.1f}s')
print('Training complete!')

SyntaxError: invalid syntax (3810330861.py, line 1)

## Section 2: Model Implementation

In [14]:
import matplotlib.pyplot as plt
from tqdm import tqdm

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()

anomaly_scores = []
poa_scores = []

print('Running inference...')
with torch.no_grad():
    for w in tqdm(windows_norm, desc='Scoring windows'):
        x = torch.FloatTensor(w['x']).unsqueeze(0).to(device)
        delta_t = torch.FloatTensor(w['delta_t']).unsqueeze(0).to(device)
        t_abs = torch.LongTensor(w['t_abs']).to(device)
        tokens = model.embed_window(x, delta_t, t_abs)
        h = model.transformer_forward(tokens, branch='anomaly')
        # Apply sigmoid to convert logits to probabilities
        anomaly_scores.append(torch.sigmoid(model.anomaly_head(h)).item())
        poa_scores.append(torch.sigmoid(model.poa_head(h)).item())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history['train'], label='Train')
axes[0, 0].plot(history['val'], label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training History')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(anomaly_scores, bins=50, alpha=0.7)
axes[0, 1].axvline(np.percentile(anomaly_scores, ANOMALY_PERCENTILE), color='r', linestyle='--', label=f'{ANOMALY_PERCENTILE}th percentile')
axes[0, 1].set_xlabel('Anomaly Score')
axes[0, 1].set_title('Score Distribution')
axes[0, 1].legend()

axes[1, 0].plot(anomaly_scores, alpha=0.7, linewidth=0.5)
axes[1, 0].axhline(np.percentile(anomaly_scores, ANOMALY_PERCENTILE), color='r', linestyle='--')
axes[1, 0].set_xlabel('Window')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Scores Over Time')

axes[1, 1].scatter(anomaly_scores, poa_scores, alpha=0.3, s=5)
axes[1, 1].set_xlabel('Anomaly Score')
axes[1, 1].set_ylabel('PoA Score')
axes[1, 1].set_title('Anomaly vs PoA')

plt.tight_layout()
plt.savefig('anomaly_detection_results.png', dpi=150)
plt.show()

# Metrics
threshold = np.percentile(anomaly_scores, ANOMALY_PERCENTILE)
anomaly_flagged = sum(1 for s in anomaly_scores if s > threshold)

precision = anomaly_flagged / len(anomaly_scores)
recall = anomaly_flagged / len(anomaly_scores)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\n=== Metrics ===')
print(f'Anomalies detected: {anomaly_flagged}/{len(anomaly_scores)} ({100*anomaly_flagged/len(anomaly_scores):.1f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'\nPlots saved to anomaly_detection_results.png')

SyntaxError: invalid syntax (1540346147.py, line 1)

## Section 3: Training

In [15]:
from tqdm import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

class PADDataset(torch.utils.data.Dataset):
    def __init__(self, windows_norm):
        self.windows = windows_norm
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return {
            'x': torch.FloatTensor(w['x']),
            'delta_t': torch.FloatTensor(w['delta_t']),
            't_abs': torch.LongTensor(w['t_abs'])
        }

dataset = PADDataset(windows_norm)
train_size = int(TRAIN_RATIO * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True if device.type == 'cuda' else False)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True if device.type == 'cuda' else False)

model = PADTransformer(
    input_dim=wide_df.shape[1],
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    use_shared=USE_SHARED
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Train: {train_size}, Val: {val_size}')

history = {'train': [], 'val': [], 'lr': [], 'epoch_time': []}
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=False)
    
    for batch in train_pbar:
        w_i = {
            'x': batch['x'].to(device),
            'delta_t': batch['delta_t'].to(device),
            't_abs': batch['t_abs'].to(device)
        }
        w_i_plus_1 = w_i
        
        y_hat_a_i, y_hat_a_i_plus_1, y_hat_p_i_plus_1 = model(w_i, w_i_plus_1)
        
        L_a = F.binary_cross_entropy(y_hat_a_i.squeeze(), torch.zeros_like(y_hat_a_i.squeeze()))
        L_kd = F.binary_cross_entropy(y_hat_a_i_plus_1.squeeze(), y_hat_p_i_plus_1.squeeze().detach())
        loss = L_a + L_kd
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]', leave=False)
    
    with torch.no_grad():
        for batch in val_pbar:
            w_i = {
                'x': batch['x'].to(device),
                'delta_t': batch['delta_t'].to(device),
                't_abs': batch['t_abs'].to(device)
            }
            w_i_plus_1 = w_i
            y_hat_a_i, y_hat_a_i_plus_1, y_hat_p_i_plus_1 = model(w_i, w_i_plus_1)
            L_a = F.binary_cross_entropy(y_hat_a_i.squeeze(), torch.zeros_like(y_hat_a_i.squeeze()))
            L_kd = F.binary_cross_entropy(y_hat_a_i_plus_1.squeeze(), y_hat_p_i_plus_1.squeeze().detach())
            batch_loss = (L_a + L_kd).item()
            val_loss += batch_loss
            val_pbar.set_postfix({'loss': f'{batch_loss:.4f}'})
    
    val_loss /= len(val_loader)
    epoch_time = time.time() - epoch_start
    
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['lr'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s | *BEST*')
    else:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s')

print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Total training time: {sum(history["epoch_time"]):.1f}s')
print('Training complete!')

Using device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
Memory: 8.19 GB
Model parameters: 817,538
Train: 5136, Val: 1284


Epoch 1/300 | Train: 1.1533 | Val: 1.1374 | LR: 1.00e-04 | Time: 4.4s | *BEST*


Epoch 2/300 | Train: 1.1384 | Val: 1.1350 | LR: 1.00e-04 | Time: 4.1s | *BEST*


Epoch 3/300 | Train: 1.1474 | Val: 1.1495 | LR: 1.00e-04 | Time: 4.1s


Epoch 4/300 | Train: 1.1548 | Val: 1.1508 | LR: 1.00e-04 | Time: 4.1s


Epoch 5/300 | Train: 1.1310 | Val: 1.1210 | LR: 1.00e-04 | Time: 4.1s | *BEST*


Epoch 6/300 [Train]:  89%|███████████████████████████████████████     | 72/81 [00:03<00:00, 21.31it/s, loss=1.1034]/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [32,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [33,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [34,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [35,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [36,0,0] Assertion `input_val >= zero && input_val <= one` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [37,0,0] Assertion `input_val >= zero && input_val <

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Section 4: Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()

anomaly_scores = []
poa_scores = []

print('Running inference...')
with torch.no_grad():
    for w in tqdm(windows_norm, desc='Scoring windows'):
        x = torch.FloatTensor(w['x']).unsqueeze(0).to(device)
        delta_t = torch.FloatTensor(w['delta_t']).unsqueeze(0).to(device)
        t_abs = torch.LongTensor(w['t_abs']).to(device)
        tokens = model.embed_window(x, delta_t, t_abs)
        h = model.transformer_forward(tokens, branch='anomaly')
        anomaly_scores.append(torch.sigmoid(model.anomaly_head(h)).item())
        poa_scores.append(torch.sigmoid(model.poa_head(h)).item())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history['train'], label='Train')
axes[0, 0].plot(history['val'], label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training History')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(anomaly_scores, bins=50, alpha=0.7)
axes[0, 1].axvline(np.percentile(anomaly_scores, ANOMALY_PERCENTILE), color='r', linestyle='--', label=f'{ANOMALY_PERCENTILE}th percentile')
axes[0, 1].set_xlabel('Anomaly Score')
axes[0, 1].set_title('Score Distribution')
axes[0, 1].legend()

axes[1, 0].plot(anomaly_scores, alpha=0.7, linewidth=0.5)
axes[1, 0].axhline(np.percentile(anomaly_scores, ANOMALY_PERCENTILE), color='r', linestyle='--')
axes[1, 0].set_xlabel('Window')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Scores Over Time')

axes[1, 1].scatter(anomaly_scores, poa_scores, alpha=0.3, s=5)
axes[1, 1].set_xlabel('Anomaly Score')
axes[1, 1].set_ylabel('PoA Score')
axes[1, 1].set_title('Anomaly vs PoA')

plt.tight_layout()
plt.savefig('anomaly_detection_results.png', dpi=150)
plt.show()

# Metrics
threshold = np.percentile(anomaly_scores, ANOMALY_PERCENTILE)
anomaly_flagged = sum(1 for s in anomaly_scores if s > threshold)

precision = anomaly_flagged / len(anomaly_scores)
recall = anomaly_flagged / len(anomaly_scores)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\n=== Metrics ===')
print(f'Anomalies detected: {anomaly_flagged}/{len(anomaly_scores)} ({100*anomaly_flagged/len(anomaly_scores):.1f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'\nPlots saved to anomaly_detection_results.png')